<a href="https://colab.research.google.com/github/ivadapally/finetuning/blob/main/Copy_of_non_instruction_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 1 — Non-Instruction Fine-Tuning

**Domain:** Customer Support Assistant
**Base model:** `unsloth/Qwen2.5-0.5B` (plain base checkpoint)

This notebook adapts the base model to customer-support *language and terminology* by training it as a plain causal language model on raw domain text (`data/non_instruction_data.txt`) — no instructions, no Q&A framing, just next-token prediction on knowledge-base-style prose about refunds, order tracking, cancellations, payments, etc.

Run this notebook on a GPU runtime (Google Colab's free T4 is enough for a 0.5B model in 4-bit).


## 1. Install dependencies

In [2]:
%%capture
!pip install -q unsloth


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


GridBox(children=(Dropdown(layout=Layout(width='auto'), options=('google/gemini-2.5-flash-lite', 'google/gemin…

In [3]:
!nvidia-smi


Sun Jul  5 01:07:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Load the base model with Unsloth (4-bit)

In [ ]:
# Force reinstall PyArrow with the correct version
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "pyarrow"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-cache-dir", "pyarrow==14.0.1"])

0

In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None          # auto-detect (bfloat16 on Ampere+, else float16)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print(model.config._name_or_path)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/521M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
unsloth/qwen2.5-0.5b-unsloth-bnb-4bit


## 3. Load, clean and chunk the raw domain text

In [6]:
from pathlib import Path

DATA_PATH = Path("/content/non_instruction_data.txt")
raw_text = DATA_PATH.read_text(encoding="utf-8")

# The file is already one clean paragraph per blank-line-separated block
# (see scripts/build_datasets.py) - split, strip, drop anything empty.
paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
print(f"Loaded {len(paragraphs)} paragraphs")
print(paragraphs[0][:300], "...")


Loaded 54 paragraphs
[CANCEL ORDER] I truly understand that you're experiencing difficulties in canceling your purchase with the order number ORD-58204. I apologize for any inconvenience this has caused you. Our team is committed to resolving this issue and assisting you in the best possible way. To address the problems ...


In [7]:
def chunk_paragraphs(paragraphs, tokenizer, block_size=512):
    """Greedily pack paragraphs into ~block_size-token chunks so each
    training example is a reasonably dense block of raw domain text rather
    than one short paragraph per row."""
    chunks, buffer = [], ""
    for p in paragraphs:
        candidate = (buffer + "\n\n" + p).strip() if buffer else p
        if buffer and len(tokenizer(candidate)["input_ids"]) > block_size:
            chunks.append(buffer)
            buffer = p
        else:
            buffer = candidate
    if buffer:
        chunks.append(buffer)
    return chunks


chunks = chunk_paragraphs(paragraphs, tokenizer, block_size=512)
print(f"Built {len(chunks)} training chunks from {len(paragraphs)} paragraphs")


Built 53 training chunks from 54 paragraphs


In [8]:
from datasets import Dataset

raw_dataset = Dataset.from_dict({"text": chunks})
raw_dataset


Dataset({
    features: ['text'],
    num_rows: 53
})

## 4. Apply LoRA

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
model.print_trainable_parameters()


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.9 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


## 5. Train on raw domain text (plain causal-LM objective, no instruction formatting)

In [10]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = raw_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "../outputs/stage1_checkpoints",
        report_to = "none",
    ),
)
trainer_stats = trainer.train()


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/53 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 53 | Num Epochs = 3 | Total steps = 21
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,1.460807
10,1.339919
15,1.273011
20,1.187173


Unsloth: Restored added_tokens_decoder metadata in ../outputs/stage1_checkpoints/checkpoint-21/tokenizer_config.json.


## 6. Save the Stage 1 adapter

In [11]:
import os

os.makedirs("../outputs", exist_ok=True)
model.save_pretrained("../outputs/stage1_adapter")
tokenizer.save_pretrained("../outputs/stage1_adapter")
print("Saved Stage 1 non-instruction adapter to ../outputs/stage1_adapter")


Unsloth: Restored added_tokens_decoder metadata in ../outputs/stage1_adapter/tokenizer_config.json.


Saved Stage 1 non-instruction adapter to ../outputs/stage1_adapter


## 7. Quick sanity test after Stage 1

In [12]:
FastLanguageModel.for_inference(model)

for prompt in ["Our refund policy", "To track your order,", "If your payment failed,"]:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=60, use_cache=True, do_sample=False)
    print("PROMPT:", prompt)
    print("CONTINUATION:", tokenizer.decode(output[0], skip_special_tokens=True))
    print("---")


Both `max_new_tokens` (=60) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=60

PROMPT: Our refund policy
CONTINUATION: Our refund policy is simple: if you're not satisfied with your purchase, you can return it within 30 days of receipt. To do this, simply follow these steps: 1. Go to the website where you purchased the item. 2. Look for the "Return" or "Refund"
---


Both `max_new_tokens` (=60) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: To track your order,
CONTINUATION: To track your order, you can use the following methods: 1. Visit the website's order page: You can find the order page on the website's homepage or by searching for it using the search bar. 2. Contact the customer support team: If you have any questions or concerns about your order, you can
---
PROMPT: If your payment failed,
CONTINUATION: If your payment failed, you can try the following steps to resolve the issue: 1. Check your account balance: Make sure you have enough funds in your account to cover the payment. If you don't have enough funds, you can try resetting your password or contacting customer support for assistance. 2. Contact your bank
---


## Next step

Continue with `instruction_finetuning.ipynb`. It will detect `../outputs/stage1_adapter` and resume from it automatically before Stage 2 (instruction) fine-tuning.
